# Notebook 3 — SQUID Flux Tunability and the Sweet Spot

A symmetric SQUID replaces the single Josephson junction of a fixed-frequency transmon with **two** junctions in a loop. The effective Josephson energy then depends on the external magnetic flux $\Phi_\mathrm{ext}$ threading the loop:

$$
E_J(\Phi_\mathrm{ext}) \;=\; E_{J,\Sigma} \, \left| \cos\!\left( \pi \, \Phi_\mathrm{ext} / \Phi_0 \right) \right|
$$

where $\Phi_0 = h / 2e$ is the magnetic flux quantum and $E_{J,\Sigma}$ is the sum of the two junction Josephson energies.

For an asymmetric SQUID (junction asymmetry $d = (E_{J,2} - E_{J,1})/(E_{J,2} + E_{J,1})$):

$$
E_J(\Phi_\mathrm{ext}) \;=\; E_{J,\Sigma} \, \sqrt{ \cos^2\!\left( \pi \Phi_\mathrm{ext} / \Phi_0 \right) + d^2 \, \sin^2\!\left( \pi \Phi_\mathrm{ext}/\Phi_0 \right) }
$$

Because the transmon qubit frequency scales as $\omega_{01} \approx \sqrt{8 E_J E_C}/\hbar - E_C/\hbar$, modulating $E_J$ via $\Phi_\mathrm{ext}$ tunes the qubit frequency. This is the hardware mechanism behind:

- **Flux-controlled iSWAP gates** (Notebook 4a — DC flux pulse to bring two qubits into resonance),
- **Parametric gates** (modulating $\Phi_\mathrm{ext}$ at a chosen frequency),
- **Tunable couplers** (a third transmon whose frequency is flux-tuned to switch coupling on and off).

## What this notebook shows

1. The cosine modulation of $E_J(\Phi)$ for a symmetric SQUID and several asymmetric cases.
2. The resulting qubit frequency $\omega_{01}(\Phi)$ — including the **sweet spot** at $\Phi=0$ where $\partial \omega_{01}/\partial \Phi = 0$ to first order.
3. The flux-noise sensitivity $|\partial \omega_{01}/\partial \Phi|$ — vanishing at the sweet spot, large away from it. Operating at the sweet spot is why fixed-frequency-style transmons (or asymmetric SQUIDs at their second sweet spot) have much longer dephasing times than mid-range-tuned ones.

All energies in units of $E_C$, $\hbar = 1$, fluxes in units of $\Phi_0$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


def EJ_of_flux(phi, EJ_sigma, d=0.0):
    """
    SQUID effective Josephson energy as a function of external flux.

    phi      : flux in units of Phi_0 (so phi = 0.5 corresponds to half-flux quantum)
    EJ_sigma : sum E_{J,1} + E_{J,2}, in units of E_C
    d        : junction asymmetry, in [0, 1].  d = 0 -> symmetric SQUID.

    Returns E_J(phi) in units of E_C.
    """
    return EJ_sigma * np.sqrt(np.cos(np.pi * phi) ** 2
                              + d ** 2 * np.sin(np.pi * phi) ** 2)


def omega01_perturbative(EJ_over_EC):
    """
    Perturbative transmon transition frequency (E_C = hbar = 1):
      omega_01 / E_C = sqrt(8 EJ/EC) - 1.
    """
    return np.sqrt(8 * EJ_over_EC) - 1.0


# Sanity check
print(f"E_J(phi=0,   d=0)   / E_sigma = {EJ_of_flux(0.0, 1.0, d=0.0):.4f}   (expect 1.0)")
print(f"E_J(phi=0.5, d=0)   / E_sigma = {EJ_of_flux(0.5, 1.0, d=0.0):.4f}   (expect 0.0)")
print(f"E_J(phi=0.5, d=0.1) / E_sigma = {EJ_of_flux(0.5, 1.0, d=0.1):.4f}   (expect 0.1)")

E_J(phi=0,   d=0)   / E_sigma = 1.0000   (expect 1.0)
E_J(phi=0.5, d=0)   / E_sigma = 0.0000   (expect 0.0)
E_J(phi=0.5, d=0.1) / E_sigma = 0.1000   (expect 0.1)


In [2]:
# Realistic transmon parameters: E_J/E_C ~ 50 at phi = 0
EJ_sigma_over_EC = 50.0

# Flux sweep over one full period
phi_grid = np.linspace(-1.0, 1.0, 801)

# Several asymmetry values, including the symmetric case
asymmetry_values = [0.0, 0.1, 0.3, 0.5]

# Storage for E_J(phi) and omega_01(phi)
EJ_curves     = {}
omega_curves  = {}

for d in asymmetry_values:
    EJ        = EJ_of_flux(phi_grid, EJ_sigma_over_EC, d=d)
    omega     = omega01_perturbative(EJ)
    EJ_curves[d]    = EJ
    omega_curves[d] = omega

print(f"Computed flux sweep for {len(asymmetry_values)} asymmetry values, "
      f"{len(phi_grid)} flux points each.")
print(f"At phi = 0 (sweet spot):")
for d in asymmetry_values:
    i0 = len(phi_grid) // 2          # phi = 0
    print(f"  d = {d:.1f}:   E_J = {EJ_curves[d][i0]:.2f} E_C,   "
          f"omega_01 = {omega_curves[d][i0]:.3f} E_C")

print(f"\nAt phi = 0.5 (anti-sweet spot for symmetric SQUID):")
for d in asymmetry_values:
    i_half = np.argmin(np.abs(phi_grid - 0.5))
    print(f"  d = {d:.1f}:   E_J = {EJ_curves[d][i_half]:.2f} E_C,   "
          f"omega_01 = {omega_curves[d][i_half]:.3f} E_C")

Computed flux sweep for 4 asymmetry values, 801 flux points each.
At phi = 0 (sweet spot):
  d = 0.0:   E_J = 50.00 E_C,   omega_01 = 19.000 E_C
  d = 0.1:   E_J = 50.00 E_C,   omega_01 = 19.000 E_C
  d = 0.3:   E_J = 50.00 E_C,   omega_01 = 19.000 E_C
  d = 0.5:   E_J = 50.00 E_C,   omega_01 = 19.000 E_C

At phi = 0.5 (anti-sweet spot for symmetric SQUID):
  d = 0.0:   E_J = 0.00 E_C,   omega_01 = -1.000 E_C
  d = 0.1:   E_J = 5.00 E_C,   omega_01 = 5.325 E_C
  d = 0.3:   E_J = 15.00 E_C,   omega_01 = 9.954 E_C
  d = 0.5:   E_J = 25.00 E_C,   omega_01 = 13.142 E_C
